In [2]:
import pandas as pd
import numpy as np

# 1. Load the datasets
movies = pd.read_csv('tmdb_5000_movies.csv')
credits = pd.read_csv('tmdb_5000_credits.csv')

# 2. Merge them on 'title'
movies = movies.merge(credits, on='title')

# 3. Select relevant columns
# We only need features that help describe the 'content'
movies = movies[['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew']]

print(movies.head())

   movie_id                                     title  \
0     19995                                    Avatar   
1       285  Pirates of the Caribbean: At World's End   
2    206647                                   Spectre   
3     49026                     The Dark Knight Rises   
4     49529                               John Carter   

                                            overview  \
0  In the 22nd century, a paraplegic Marine is di...   
1  Captain Barbossa, long believed to be dead, ha...   
2  A cryptic message from Bond’s past sends him o...   
3  Following the death of District Attorney Harve...   
4  John Carter is a war-weary, former military ca...   

                                              genres  \
0  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   
1  [{"id": 12, "name": "Adventure"}, {"id": 14, "...   
2  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   
3  [{"id": 28, "name": "Action"}, {"id": 80, "nam...   
4  [{"id": 28, "name": "Action"}, {"id":

In [6]:
import ast

def convert(obj):
    L = []
    # If the object is a string, try to evaluate it into a list/dict
    if isinstance(obj, str):
        try:
            obj = ast.literal_eval(obj)
        except (ValueError, SyntaxError):
            return L # Return empty list if string is malformed
            
    # Now that it's a list/dict, we can iterate
    if isinstance(obj, list):
        for i in obj:
            # Check if it's a list of dictionaries (standard TMDB format)
            if isinstance(i, dict) and 'name' in i:
                L.append(i['name'])
            # Check if it's already a list of strings (your current state)
            elif isinstance(i, str):
                L.append(i)
    return L

# Apply the updated function
movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)

In [7]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Convert tags into a matrix of token counts
cv = CountVectorizer(max_features=5000, stop_words='english')
vectors = cv.fit_transform(new_df['tags']).toarray()

# 2. Calculate Similarity between all movies
similarity = cosine_similarity(vectors)

# 3. The Recommendation Function
def recommend(movie):
    movie_index = new_df[new_df['title'] == movie].index[0]
    distances = similarity[movie_index]
    movies_list = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[1:6]
    
    print(f"Recommendations for '{movie}':")
    for i in movies_list:
        print(new_df.iloc[i[0]].title)

# Test it out!
recommend('Avatar')

Recommendations for 'Avatar':
Aliens
Mission to Mars
Moonraker
Silent Running
Spaceballs


In [8]:
import pickle

# Save the movie dataframe (as a dictionary for easier loading)
pickle.dump(new_df.to_dict(), open('movie_dict.pkl', 'wb'))

# Save the similarity matrix
pickle.dump(similarity, open('similarity.pkl', 'wb'))